# Demo: Carregando o Modelo Production do MLflow

Este notebook demonstra como:

1. Conectar ao **MLflow** (tracking SQLite local)
2. Buscar o **melhor modelo NCF** em todos os experimentos
3. **Carregar o modelo treinado** e gerar uma recomendação de exemplo

**Modelo Production**: `Ablation_FINAL_no_aux_emb32` (NDCG@10 = 0.2725)

---

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import torch
import mlflow
from mlflow.tracking import MlflowClient
import yaml

from src.config import settings
from src.models.ncf import NCFHybrid
from src.data.preprocessing import fit_scaler, transform_features

# Conectando ao tracking local (SQLite, conforme src/config.py)
# Usar path absoluto (relativo ao PROJECT_ROOT) para funcionar independente do CWD
mlflow.set_tracking_uri(f'sqlite:///{PROJECT_ROOT}/artifacts/mlflow.db')
print(f'Tracking URI: {settings.mlflow_uri}')
print(f'Experiment default: {settings.experiment_name}')

client = MlflowClient()
print(f'\nExperimentos disponiveis:')
for exp in client.search_experiments():
    print(f'  - {exp.experiment_id}: {exp.name}')


/home/bill/Codes/ML_Eng_Projects/pos-ml-eng-tech-challenge-fase-02/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tracking URI: sqlite:///./artifacts/mlflow.db
Experiment default: Olist_Recommender



Experimentos disponiveis:
  - 4: Olist_Recommender
  - 3: Olist_NCF_Ablation
  - 2: Olist_NCF_Optimization
  - 1: Olist_NCF_Baseline
  - 0: Default


## 1. Encontrar o Melhor Modelo NCF

In [2]:
# Buscar a melhor run NCF em todos os experimentos (exceto Default).
# Criterio: maior test_NDCG_at_K.
all_experiments = [
    exp for exp in client.search_experiments()
    if exp.name != 'Default'
]
exp_ids = [e.experiment_id for e in all_experiments]

# Filtra runs com test_NDCG_at_K > 0 manualmente (SQLite backend)
all_finished = client.search_runs(
    experiment_ids=exp_ids,
    filter_string="status = 'FINISHED'",
    order_by=["metrics.test_NDCG_at_K DESC"],
    max_results=50,
)
best_runs = [r for r in all_finished if r.data.metrics.get('test_NDCG_at_K', 0) > 0][:10]

if not best_runs:
    raise RuntimeError('Nenhuma run FINALIZADA com test_NDCG_at_K > 0 encontrada.')

best_run = best_runs[0]
print(f'Melhor modelo: {best_run.info.run_name}')
print(f'  Run ID:       {best_run.info.run_id}')
print(f'  Experiment:   {best_run.info.experiment_id}')
print(f'  test_NDCG@K:  {best_run.data.metrics["test_NDCG_at_K"]:.4f}')
print(f'  test_HitRate: {best_run.data.metrics.get("test_HitRate_at_K", 0):.4f}')
print(f'  test_Recall:  {best_run.data.metrics.get("test_Recall_at_K", 0):.4f}')
print(f'  Params:')
for k, v in best_run.data.params.items():
    print(f'    {k}: {v}')


Melhor modelo: Ablation_FINAL_no_aux_emb32
  Run ID:       a905125600df4452a2d3f3581a87ab42
  Experiment:   3
  test_NDCG@K:  0.2725
  test_HitRate: 0.4949
  test_Recall:  0.4886
  Params:
    emb_dim: 32
    hidden: [64, 32]
    dropout: 0.5
    lr: 0.0005
    batch_size: 2048
    n_negatives: 8
    epochs: 20
    n_params: 4027009
    n_aux_features: 20


## 2. Carregar o Modelo Production e os Artefatos

In [3]:
# Carregar config e o scaler
with open(PROJECT_ROOT / 'configs' / 'selected_features.yaml') as f:
    config = yaml.safe_load(f)

# Construir o modelo com a mesma arquitetura do Production (Ablation_FINAL_no_aux_emb32)
# O checkpoint tem input dim 92 = 32 (user) + 32 (item) + 8 (cat) + 20 (aux originais).
PRODUCTION_NUMERIC_COLS = [
    'price_log', 'freight_value_log', 'price_to_freight_ratio',
    'has_price_outlier', 'days_since_reference', 'is_weekend',
    'is_holiday_season', 'category_target_enc', 'category_frequency',
    'payment_type_credit_card', 'payment_type_boleto',
    'payment_type_voucher', 'payment_type_debit_card',
    'user_total_purchases', 'user_avg_freight', 'user_purchase_span_days',
    'user_recency_days', 'product_popularity', 'product_avg_review_score',
    'product_review_rate',
]
model = NCFHybrid(
    n_users=config['n_users'],
    n_items=config['n_items'],
    n_categories=config['n_categories'],
    n_aux_features=20,
    emb_dim=32,
    cat_emb_dim=8,
    hidden=[64, 32],
    dropout=0.5,
)
production_model_path = PROJECT_ROOT / 'artifacts' / 'ncf_Ablation_FINAL_no_aux_emb32.pt'
state_dict = torch.load(production_model_path, map_location='cpu', weights_only=False)
model.load_state_dict(state_dict)
model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'Modelo Production carregado de {production_model_path.name}')
print(f'Parametros: {n_params:,}')

# Carregar scaler fitado no treino
df = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'interactions_fe.parquet')
from src.data.splits import temporal_split
train_df, _, _ = temporal_split(df, time_col='days_since_reference', descending=False)
scaler = fit_scaler(train_df, PRODUCTION_NUMERIC_COLS)
print(f'Scaler fitado em {len(train_df):,} linhas de treino')


Modelo Production carregado de ncf_Ablation_FINAL_no_aux_emb32.pt
Parametros: 4,027,009
Treino: 69,849 registros (até 577)
Validação: 14,968 registros (577 → 643)
Teste: 14,968 registros (a partir de 643)
Scaler fitado em 69,849 linhas de treino


## 3. Gerar Recomendacao de Exemplo para um Usuario

In [4]:
# Escolher um usuario warm-start (com historico no treino)
from src.data.dataset import build_user_items_map
user_items_map = build_user_items_map(train_df)
all_item_ids = df['product_id_idx'].unique()

warm_users = [u for u, items in user_items_map.items() if len(items) >= 3]
warm_user = max(warm_users, key=lambda u: len(user_items_map[u]))
history = user_items_map[warm_user]
print(f'Usuario warm-start: {warm_user}')
print(f'Itens no historico de treino: {len(history)}')

# Gerar candidatos: 500 itens nao vistos
rng = np.random.default_rng(42)
candidates = []
attempts = 0
while len(candidates) < 500 and attempts < 5000:
    item = int(rng.choice(all_item_ids))
    if item not in history and item not in candidates:
        candidates.append(item)
    attempts += 1
print(f'Candidatos amostrados: {len(candidates)}')

# Construir features dos candidatos via train_df
item_lookup = train_df.drop_duplicates('product_id_idx').set_index('product_id_idx')
cats = [int(item_lookup.loc[c, 'category_id']) if c in item_lookup.index else 0 for c in candidates]
aux_rows = []
for c in candidates:
    if c in item_lookup.index:
        aux_rows.append(transform_features(item_lookup.loc[[c]], scaler, PRODUCTION_NUMERIC_COLS)[0])
    else:
        aux_rows.append(np.zeros(len(PRODUCTION_NUMERIC_COLS), dtype=np.float32))

# Scoring em batch
with torch.no_grad():
    user_t = torch.full((len(candidates),), warm_user, dtype=torch.long)
    item_t = torch.tensor(candidates, dtype=torch.long)
    cat_t = torch.tensor(cats, dtype=torch.long)
    aux_t = torch.tensor(np.stack(aux_rows), dtype=torch.float32)
    scores = model(user_t, item_t, cat_t, aux_t).numpy()

top10_idx = np.argsort(scores)[-10:][::-1]
top10_items = [candidates[i] for i in top10_idx]
top10_scores = scores[top10_idx]

# Enriquecer com categoria e popularidade
rec_info = train_df[train_df['product_id_idx'].isin(top10_items)][
    ['product_id_idx', 'category_id', 'product_popularity']
].drop_duplicates('product_id_idx').set_index('product_id_idx')

print('\nTop-10 Recomendacoes:')
print('-' * 60)
for rank, (item, score) in enumerate(zip(top10_items, top10_scores), 1):
    cat = rec_info.loc[item, 'category_id'] if item in rec_info.index else '?'
    pop = rec_info.loc[item, 'product_popularity'] if item in rec_info.index else '?'
    print(f'  {rank:2d}. Item {item:6d} | cat={cat:3} | pop={pop:4} | score={score:+.4f}')


Usuario warm-start: 22779
Itens no historico de treino: 13
Candidatos amostrados: 500



Top-10 Recomendacoes:
------------------------------------------------------------
   1. Item  29793 | cat= 12 | pop=   1 | score=+10.5791
   2. Item  13215 | cat= 34 | pop=   3 | score=+10.5701
   3. Item   3034 | cat= 30 | pop=  22 | score=+10.4446
   4. Item   6531 | cat= 16 | pop=   1 | score=+10.3808
   5. Item    448 | cat= 20 | pop=   1 | score=+10.0616
   6. Item  15678 | cat= 20 | pop=   1 | score=+10.0237
   7. Item  23302 | cat= 31 | pop=   1 | score=+9.6052
   8. Item  12054 | cat= 14 | pop=   2 | score=+9.3761
   9. Item  20288 | cat= 29 | pop=   1 | score=+8.1943
  10. Item  29270 | cat= 15 | pop=   1 | score=+8.0957


## 4. Resumo

Este notebook demonstrou o fluxo end-to-end de uso do modelo em producao:

1. **MLflow tracking** (SQLite local) registra todas as runs com metricas, parametros e artefatos.
2. **MLflow client** permite buscar o melhor modelo via query SQL-like.
3. **Modelo carregado** do checkpoint `.pt` salvo em `artifacts/`.
4. **Scaler** re-fitado no treino garante consistencia das features.
5. **Scoring em batch** (500 candidatos) retorna o top-10 ordenado por score.

Em producao, este fluxo seria substituido por um servico HTTP (ex: FastAPI) que carrega
o modelo uma unica vez na inicializacao e responde requests de recomendacao em < 100ms.